In [ ]:
# Colab setup: install packages not preinstalled on Colab (safe to re-run)
!pip install -q abess

# Reproduce-then-Extend — Election-Law Reform after the 2000 Election (American State Politics)  ·  **Day 1 tutorial**

> **Published study.** Palazzolo, D. J. & Moscardelli, V. G. (2006). "Policy Crisis and Political Leadership:
> Election Law Reform in the States after the 2000 Presidential Election." *State Politics & Policy Quarterly*
> 6(3):300–321 (doi:10.1177/153244000600600303). Replication data: UNC SPPQ Dataverse doi:10.15139/S3/12145.
> The published model is a **cross-sectional OLS** of each state's election-reform activity on **13**
> predictors with only **50 states** — a genuinely high predictor-to-observation ratio, which is exactly the
> setting where **regularization and variable selection** earn their keep.

## Background

The disputed 2000 presidential election (the Florida recount, "hanging chads," *Bush v. Gore*) was a **focusing
event** that put election administration on every state's agenda. Yet states responded very differently — some
overhauled their voting systems and registration rules, others did little. Palazzolo & Moscardelli ask **what
explains the variation in states' election-law reform activity**, and argue that **political leadership** —
not just the objective severity of a state's voting problems — drove reform. They test this with an ordinary
cross-sectional regression of a weighted count of reforms on a leadership index plus political, institutional,
and problem-severity controls.

## Data and codebook

**Unit of analysis:** a U.S. **state**; n = 50 (the published model excludes **Florida** as the 2000 outlier,
leaving 49). Data are the authors' SPPQ replication file.

| Variable | Definition |
|---|---|
| `reformwt` | weighted count of election reforms adopted **(outcome)** |
| `leadindx` | **leadership index** (legislative/executive reform leadership) |
| `ranney` | party competitiveness (folded Ranney index) |
| `simple` | simple divided government |
| `compound` | compound divided government |
| `termrank` | legislative term limits |
| `culture` | political culture |
| `ideology` | conservative state ideology (Norrander) |
| `fiscalin` | ratio of state revenues to expenditures (fiscal slack) |
| `lwv_act` | interest-group mobilization (League of Women Voters activity) |
| `ffactor` | winner's margin in the 2000 presidential election |
| `residvot` | residual (uncounted) vote rate in 2000 — a measure of the state's voting problems |
| `ffrxrv` | interaction of the (recoded) 2000 margin and the residual-vote rate |
| `commrec` | number of reform-commission recommendations |

## Descriptive results

In [ ]:
import warnings; warnings.filterwarnings('ignore')
import numpy as np, pandas as pd, matplotlib.pyplot as plt
import statsmodels.api as sm
from sklearn.linear_model import LassoCV, RidgeCV, ElasticNetCV, LinearRegression, lasso_path
from sklearn.preprocessing import StandardScaler
from sklearn.model_selection import train_test_split
from sklearn.metrics import mean_squared_error
rng = np.random.RandomState(2026)

In [ ]:
d = pd.read_csv('https://raw.githubusercontent.com/desmarais-lab/desmarais-lab.github.io/master/istanbul_bilgi_ml_files/data/election_law_reform.csv')
d = d[d['state'] != 9]                                  # exclude Florida (the 2000 outlier), as the paper does
outcome = 'reformwt'
preds = ['leadindx','ranney','simple','compound','termrank','culture','ideology',
         'fiscalin','lwv_act','ffactor','residvot','ffrxrv','commrec']
print(f'{d.shape[0]} states x {len(preds)} predictors (p/n = {len(preds)/d.shape[0]:.2f})')
d[[outcome]+preds].describe().T.round(2)

In [ ]:
fig, ax = plt.subplots(1, 2, figsize=(9, 3.2))
ax[0].hist(d[outcome], bins=12, color='#a6cee3', edgecolor='white')
ax[0].set_title('Outcome: weighted reform count'); ax[0].set_xlabel('reforms')
cors = d[preds].corrwith(d[outcome]).sort_values()
ax[1].barh(cors.index, cors.values, color=['#1f78b4' if v>0 else '#e31a1c' for v in cors.values])
ax[1].axvline(0, color='grey'); ax[1].set_title('Correlation with reform activity'); plt.tight_layout()

## Reproduce the published regression

The published model regresses states' weighted reform count on the leadership index and the political,
institutional, and problem-severity controls (Florida excluded).

In [ ]:
ols = sm.OLS(d[outcome], sm.add_constant(d[preds])).fit()
print(ols.summary())
print(f'\nIn-sample R^2 = {ols.rsquared:.2f}')

**Confirmation against the published study.** The model recovers the paper's central result: the
**leadership index is a positive, significant** predictor of reform activity (coefficient $\approx +0.96$,
$p \approx 0.01$), with in-sample $R^2 \approx 0.45$ — political leadership, not just the severity of a state's
voting problems, drives election-law reform. But 13 predictors on 49 states is a lot to ask of ordinary least
squares: the model fits the sample yet, as we'll see, **predicts new states poorly** — the classic symptom of
over-fitting.

## Regularization & variable selection

With **13 predictors and only 49 states** (p/n $\approx$ 0.27), OLS has few observations per coefficient and
**over-fits**. Penalized regression — lasso ($L_1$), ridge ($L_2$), elastic net — shrinks the coefficients to
trade a little bias for much lower variance, and the lasso additionally **selects** a compact subset.

In [ ]:
X = d[preds].values.astype(float); y = d[outcome].values.astype(float)
Xs = StandardScaler().fit_transform(X)
liCV = LassoCV(cv=5, random_state=0, max_iter=100000).fit(Xs, y)
n_keep = int(np.sum(liCV.coef_ != 0))
print(f'At the CV-optimal penalty the lasso keeps {n_keep} of {len(preds)} predictors (rest set to exactly 0).')
pd.DataFrame({'OLS(std)': LinearRegression().fit(Xs,y).coef_,
              'Lasso': liCV.coef_,
              'Ridge': RidgeCV(alphas=np.logspace(-2,3,40)).fit(Xs,y).coef_,
              'ElasticNet': ElasticNetCV(cv=5, l1_ratio=0.5, random_state=0, max_iter=100000).fit(Xs,y).coef_},
             index=preds).round(3)

**Which fits best out of sample?** With n = 49 a single split is noisy, so we **repeat a 70/30 split 50
times**: each replicate tunes the penalty by cross-validation on the training states and scores once on the
held-out states, reporting held-out **RMSE** (reforms).

In [ ]:
def rmse(a,b): return float(np.sqrt(mean_squared_error(a,b)))
REPS = 50; res = {k:[] for k in ['OLS','Lasso','Ridge','ElasticNet']}
for r in range(REPS):
    Xtr,Xte,ytr,yte = train_test_split(X, y, test_size=0.30, random_state=2025+r)
    sc = StandardScaler().fit(Xtr); Xtrs, Xtes = sc.transform(Xtr), sc.transform(Xte)
    res['OLS'].append(rmse(yte, LinearRegression().fit(Xtr,ytr).predict(Xte)))
    res['Lasso'].append(rmse(yte, LassoCV(cv=5,random_state=0,max_iter=100000).fit(Xtrs,ytr).predict(Xtes)))
    res['Ridge'].append(rmse(yte, RidgeCV(alphas=np.logspace(-2,3,40)).fit(Xtrs,ytr).predict(Xtes)))
    res['ElasticNet'].append(rmse(yte, ElasticNetCV(cv=5,l1_ratio=0.5,random_state=0,max_iter=100000).fit(Xtrs,ytr).predict(Xtes)))
mean = {k:np.mean(v) for k,v in res.items()}
for k,v in mean.items(): print(f'{k:12s} held-out RMSE = %.3f' % v)
best = min(['Lasso','Ridge','ElasticNet'], key=lambda k: mean[k])
print(f'\nBest penalization: {best}. {100*(mean["OLS"]-mean[best])/mean["OLS"]:.0f}% lower held-out RMSE than OLS.')

Because there are so few states per predictor, unpenalized OLS **over-fits** — its held-out error is the
worst of the four — while the penalized models **predict new states more accurately** (here the best penalty
cuts held-out RMSE by roughly 8% versus OLS), with the lasso simultaneously **selecting** the handful of
characteristics — led by the **leadership index** — that carry the signal. This is the honest lesson of a
small-sample, many-predictor design: the penalty does not add signal, it stops OLS from chasing noise.

In [ ]:
alphas, cpath, _ = lasso_path(Xs, y, n_alphas=40)
plt.figure(figsize=(6,3.4)); plt.plot(np.log10(alphas), cpath.T, color='#1f78b4', alpha=.4)
plt.xlabel('log10(alpha)  (more penalty ->)'); plt.ylabel('coefficient'); plt.title('Lasso coefficient paths'); plt.tight_layout()

## Best-subset selection with ABESS

Lasso reaches a sparse model through shrinkage. **Best-subset selection** instead searches directly for the
subset of predictors that fits best; the **adaptive best-subset (ABESS)** algorithm does so efficiently and
chooses the subset size automatically.

In [ ]:
try:
    from abess.linear import LinearRegression as AbessLR
    ab = AbessLR(support_size=range(1, 9)).fit(Xs, y)   # search subset sizes 1..8; pick the best by information criterion
    sel = [p for p,c in zip(preds, ab.coef_) if c!=0]
    print(f'ABESS selects a best subset of {len(sel)} predictor(s): {sel}')
except Exception as e:
    print('Install abess to run this cell:  !pip install abess')
    print('(', e, ')')

On this thin (49-state) design the information criterion is unforgiving: best-subset selection zeroes in
on the **single most predictive characteristic — the leadership index** — echoing Palazzolo & Moscardelli's
central claim, while the lasso retains a broader set. Both routes agree that **leadership** is the primary
driver, reached by direct search rather than shrinkage.

## Takeaway

This example illustrates **regularization in a small-sample, many-predictor design** using a real published
study. Palazzolo & Moscardelli fit 13 predictors to 49 states; ordinary regression over-fits and predicts new
states poorly, while lasso/ridge/elastic-net (`scikit-learn`) and best-subset selection (`abess`) predict more
accurately **and** name the compact set of drivers — with **political leadership** at the top — behind states'
election-law reforms after 2000.

## Recommended exercises

1. Put Florida back in and refit: how much do the OLS coefficients move, and does regularization absorb the outlier?
2. Compare the CV-optimal (`lambda.min`-style) and a sparser (1-SE-style) penalty: how much accuracy does the sparser model give up?
3. Drop the interaction `ffrxrv` and its components; does held-out error rise or fall?
4. Report which predictors ABESS and the lasso agree on, and interpret that compact model substantively.